In [1]:
# jupyter nbconvert "/Users/jasperx/Library/CloudStorage/Dropbox/1. Haiqing Xu/1. Project/Stag2_project/UltraSeq_NewLibrary_Batch1/Read-Depth-QC/Python_code/Stag2_Data_trouble_shooting_Ongoing.ipynb" --no-input --to html

<font color=red font size = 10> Modified the raw data to make sure it fit the new pipeline</font> 


# Contamination analysis 

# QC module

## 1 Functions and module

### 1.1 Modules

In [1]:
import pandas as pd

In [2]:
def label_point(x, y, val, ax):
    a = pd.concat({'x': x, 'y': y, 'val': val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point['x']+.05, point['y']+0.05, str(point['val']),size = 10)
# label_point(temp_df['TTR'], temp_df['gRNA_recovered'], temp_df['Sample_ID'], plt.gca()) # this is for labeling

In [3]:
def generate_symmetric_list(input_n):
    if input_n % 2 == 1:
        return(np.array(range(-int((input_n-1)/2),int((input_n+1)/2)))*0.1)
    else:
        return(np.array(range(-int(input_n/2),int(input_n/2)))*0.1+0.05)

----

## 2 Input and output address

In [4]:
mapping = {
    'NI': 'Lenti-Control',
    'MI': 'Lenti-mCherry',
    'HI': 'Lenti-SIINFEKL'
}

In [5]:
query_gene_list = ['Lkb1','Setd2','Trp53','Rb1','Rbm10','Apc','Arid1a','Cdkn2a','Keap1','Atm','Smad4','PDL1','B2M','Neo','NT1','NT3']

In [6]:
# old data
# combined barcode dataframe address
D1 = '/Users/jasperx/Library/CloudStorage/Dropbox/1. Haiqing Xu/1. Project/6. Immunoediting/Final_data_analysis/QC/Data/'
NI_address = D1+"Immunoediting_NI_final_df.csv"
MI_address = D1+"Immunoediting_MI_final_df.csv"
HI_address = D1+"Immunoediting_HI_final_df.csv"

In [ ]:
output_data_address = '/Users/jasperx/Library/CloudStorage/Dropbox/1. Haiqing Xu/1. Project/6. Immunoediting/Final_data_analysis/Bootstrapping/Data/'

output_data_address1 = output_data_address + "Immunoediting_NI_raw_final_df.parquet"
output_data_address2 = output_data_address + "Immunoediting_MI_raw_final_df.parquet"
output_data_address3 = output_data_address + "Immunoediting_HI_raw_final_df.parquet"

-----

## 3 Data input

In [34]:
excluded_sample = ['29536','29837', '29915']
# 29536: Too high tumor burden for the NI vector
# 29837: Get contamination from sample 29791
# 29915: dimension reduction indicate it is not good.

### 3.0 Read raw data

In [38]:
NI_raw_df = pd.read_csv(NI_address)
# Convert 'Sample_ID' column to string
NI_raw_df['Sample_ID'] = NI_raw_df['Sample_ID'].astype(str)
NI_raw_df = NI_raw_df[NI_raw_df.Identity=='gRNA']
NI_raw_df['Vector_type'] = NI_raw_df['Vector_type'].replace(mapping)
NI_raw_df = NI_raw_df[NI_raw_df.Targeted_gene_name.isin(query_gene_list)]
NI_raw_df = NI_raw_df[~NI_raw_df.Sample_ID.isin(excluded_sample)]

In [40]:
MI_raw_df = pd.read_csv(MI_address)
# Convert 'Sample_ID' column to string
MI_raw_df['Sample_ID'] = MI_raw_df['Sample_ID'].astype(str)
MI_raw_df = MI_raw_df[MI_raw_df.Identity=='gRNA']
MI_raw_df['Vector_type'] = MI_raw_df['Vector_type'].replace(mapping)
MI_raw_df = MI_raw_df[MI_raw_df.Targeted_gene_name.isin(query_gene_list)]
MI_raw_df = MI_raw_df[~MI_raw_df.Sample_ID.isin(excluded_sample)]

In [43]:
HI_raw_df = pd.read_csv(HI_address)
# Convert 'Sample_ID' column to string
HI_raw_df['Sample_ID'] = HI_raw_df['Sample_ID'].astype(str)
HI_raw_df = HI_raw_df[HI_raw_df.Identity=='gRNA']
HI_raw_df['Vector_type'] = HI_raw_df['Vector_type'].replace(mapping)
HI_raw_df = HI_raw_df[HI_raw_df.Targeted_gene_name.isin(query_gene_list)]
HI_raw_df = HI_raw_df[~HI_raw_df.Sample_ID.isin(excluded_sample)]

In [44]:
sgInert_list = ['Neo','NT1','NT3']

NI_raw_df['Type'] = 'Experiment'
NI_raw_df.loc[NI_raw_df.Targeted_gene_name.isin(sgInert_list),'Type'] = 'Inert'

MI_raw_df['Type'] = 'Experiment'
MI_raw_df.loc[MI_raw_df.Targeted_gene_name.isin(sgInert_list),'Type'] = 'Inert'

HI_raw_df['Type'] = 'Experiment'
HI_raw_df.loc[HI_raw_df.Targeted_gene_name.isin(sgInert_list),'Type'] = 'Inert'


NI_raw_df.loc[NI_raw_df.Targeted_gene_name.isin(sgInert_list),'Targeted_gene_name'] = 'Inert'
MI_raw_df.loc[MI_raw_df.Targeted_gene_name.isin(sgInert_list),'Targeted_gene_name'] = 'Inert'
HI_raw_df.loc[HI_raw_df.Targeted_gene_name.isin(sgInert_list),'Targeted_gene_name'] = 'Inert'

In [45]:
NI_raw_df.head()

,gRNA,Clonal_barcode,Sample_ID,Count,Vector_ID,Targeted_gene_name,Identity,Numbered_gene_name,Mouse_Ear_Tag,Mouse_genotype,...,Time_after_tumor_initiation,Total_lung_weight,Pooling_library_name,gRNA_clonalbarcode,Barcode_pattern,Cell_number_per_read,Correction_for_spikein,Cell_number,Vector_type,Type
717,AGACAAGCACCAGAAAGACC,AAAACCAAAA,28858,2,AGTA,B2M,gRNA,B2M_2,28858.0,KT,...,12.0,0.3189,Library 1,AGACAAGCACCAGAAAGACC_AAAACCAAAA,Real,0.705365,No,1.410729,Lenti-Control,Experiment
718,AGACAAGCACCAGAAAGACC,AAAATTTTTA,28858,337,AGTA,B2M,gRNA,B2M_2,28858.0,KT,...,12.0,0.3189,Library 1,AGACAAGCACCAGAAAGACC_AAAATTTTTA,Real,0.705365,No,237.707847,Lenti-Control,Experiment
719,AGACAAGCACCAGAAAGACC,AAACCCAAAC,28858,1083,AGTA,B2M,gRNA,B2M_2,28858.0,KT,...,12.0,0.3189,Library 1,AGACAAGCACCAGAAAGACC_AAACCCAAAC,Real,0.705365,No,763.909789,Lenti-Control,Experiment
720,AGACAAGCACCAGAAAGACC,AAATCTTTAT,28858,453,AGTA,B2M,gRNA,B2M_2,28858.0,KT,...,12.0,0.3189,Library 1,AGACAAGCACCAGAAAGACC_AAATCTTTAT,Real,0.705365,No,319.530133,Lenti-Control,Experiment
721,AGACAAGCACCAGAAAGACC,AAATTCTATA,28858,396,AGTA,B2M,gRNA,B2M_2,28858.0,KT,...,12.0,0.3189,Library 1,AGACAAGCACCAGAAAGACC_AAATTCTATA,Real,0.705365,No,279.324355,Lenti-Control,Experiment


## 4 Data generation (normal method)

In [46]:
NI_raw_df.to_parquet(output_data_address1,index=False)
MI_raw_df.to_parquet(output_data_address2,index=False)
HI_raw_df.to_parquet(output_data_address3,index=False)